# This code takes in the upset frequency files, and creates a CSV of the upset frequency per season per league.

In [5]:
# --- Compute upset frequency per league-season and save CSVs ---

import pandas as pd
from pathlib import Path

# INPUTS
IN_COMBINED = "/mnt/data/us_combined_with_upset.csv"               # real data
IN_SKILL    = "/mnt/data/us_pure_skill_with_upset.csv"            # pure-skill
IN_LUCK     = "/mnt/data/us_coin_flip_home_bias_v1_with_upset.csv" # coin-flip (luck)

# OUTPUTS
OUT_COMBINED = "/mnt/data/us_combined_upset_frequency.csv"
OUT_SKILL    = "/mnt/data/us_pure_skill_upset_frequency.csv"
OUT_LUCK     = "/mnt/data/us_coin_flip_home_bias_v1_upset_frequency.csv"

def process_file(in_path, out_path):
    df = pd.read_csv(in_path)

    # Expect exact columns: 'league', 'season', 'is_upset'
    dff = df[['league', 'season', 'is_upset']].copy()

    # Ensure 0/1 and compute frequency as mean
    dff = dff[dff['is_upset'].isin([0, 1])]
    agg = (
        dff.groupby(['league', 'season'], as_index=False)['is_upset']
           .mean()
           .rename(columns={'is_upset': 'upset_frequency'})
           .sort_values(['league', 'season'])
    )

    # Round to 4 decimals
    agg['upset_frequency'] = agg['upset_frequency'].round(4)

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    # SV stores 4 decimal digits
    agg.to_csv(out_path, index=False, float_format="%.4f")

    print(f"Processed {in_path} → {out_path}")
    # Print preview
    preview = agg.head(10).copy()
    preview['upset_frequency'] = preview['upset_frequency'].map(lambda x: f"{x:.4f}")
    print(preview.to_string(index=False))
    print('-' * 60)

process_file(IN_COMBINED, OUT_COMBINED)
process_file(IN_SKILL,    OUT_SKILL)
process_file(IN_LUCK,     OUT_LUCK)


Processed /mnt/data/us_combined_with_upset.csv → /mnt/data/us_combined_upset_frequency.csv
league  season upset_frequency
   MLB    1980          0.4138
   MLB    1981          0.4103
   MLB    1982          0.4271
   MLB    1983          0.4220
   MLB    1984          0.4413
   MLB    1985          0.4137
   MLB    1986          0.4318
   MLB    1987          0.4390
   MLB    1988          0.4138
   MLB    1989          0.4307
------------------------------------------------------------
Processed /mnt/data/us_pure_skill_with_upset.csv → /mnt/data/us_pure_skill_upset_frequency.csv
league  season upset_frequency
   MLB    1980          0.0000
   MLB    1981          0.0036
   MLB    1982          0.0000
   MLB    1983          0.0000
   MLB    1984          0.0057
   MLB    1985          0.0000
   MLB    1986          0.0000
   MLB    1987          0.0114
   MLB    1988          0.0057
   MLB    1989          0.0000
------------------------------------------------------------
Processed 